In [ ]:
from neural_lam import metrics

ROOT_DIR = "/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/280126"
EXPERIMENTS = ['SI_200', 'SI_100', 'SI_20', 'SI_10', 'CorrDiff_100', 'CorrDiff_50', 'CorrDiff_10', 'CorrDiff_5', 'EDM_100', 'EDM_50', 'EDM_10', 'EDM_5']

In [ ]:
import re
from pathlib import Path
import torch
import math
import pandas as pd
import tqdm
from neural_lam import metrics

# select device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})")   # loosen: don't require immediate ".pt"


def list_dates(expdir: Path):
    files = list(expdir.glob("*.pt"))
    dates = set()
    for f in files:
        m = DATE_RE.search(f.name)
        if m:
            dates.add(m.group(1))
    return sorted(dates)


def load_tensor(path: Path, device: torch.device = device) -> torch.Tensor:
    # load directly to device when possible
    try:
        obj = torch.load(path, map_location=device)
    except Exception:
        obj = torch.load(path, map_location="cpu")
    # handle common container formats
    if isinstance(obj, dict):
        for k in ("arr", "data", "tensor", "array"):
            if k in obj:
                obj = obj[k]
                break
        else:
            vals = [v for v in obj.values() if torch.is_tensor(v) or hasattr(v, "shape")]
            if vals:
                obj = vals[0]
            else:
                raise ValueError(f"Unsupported dict content in {path}")
    t = torch.as_tensor(obj, dtype=torch.float32, device=device)
    return t


def to_N_d(t: torch.Tensor) -> torch.Tensor:
    # ensure tensor is on chosen device
    t = t.to(device)
    if t.ndim == 0:
        return t.reshape(-1, 1)
    if t.ndim == 1:
        return t.unsqueeze(-1)
    if t.ndim == 2:
        return t.reshape(-1, 1)
    if t.ndim == 3:
        c0, d1, d2 = t.shape
        if c0 <= 4 and d1 > 1 and d2 > 1:
            t = t.permute(1, 2, 0)  # C,H,W -> H,W,C
            return t.reshape(-1, t.shape[-1])
        else:
            return t.reshape(-1, t.shape[-1])
    return t.reshape(-1, t.shape[-1])


def scalar_from_tensor(x):
    if torch.is_tensor(x):
        return float(x.item())
    return float(x)


def compute_for_experiment(expname: str):
    expdir = ROOT_DIR / expname
    if not expdir.exists():
        raise FileNotFoundError(expdir)

    dates = list_dates(expdir)
    results = []
    for date in tqdm.tqdm(dates, desc=expname):
        member_paths = sorted(expdir.glob(f"member_*_{date}.pt"))

        # robust search for target files (match date anywhere, allow extra suffixes)
        import re as _re
        target_re = _re.compile(rf"^target_.*{_re.escape(date)}.*\.pt$", flags=_re.IGNORECASE)
        target_paths = [p for p in expdir.glob("*.pt") if target_re.match(p.name)]

        if not member_paths:
            continue
        if len(target_paths) != 1:
            # debug: show what's in the folder and what would match
            available = sorted(p.name for p in expdir.glob("*.pt"))
            matched = [p.name for p in available if date in p]
            print(f"skip {expname} {date}: target files {target_paths}")
            print(f"  available .pt files count: {len(available)}; examples: {available[:10]}")
            print(f"  filenames containing date: {matched}")
            continue

        members = []
        for p in member_paths:
            t = load_tensor(p, device=device)
            members.append(to_N_d(t))

        target = to_N_d(load_tensor(target_paths[0], device=device))

        # stack ensemble: (M, N, d_state)
        try:
            preds = torch.stack(members, dim=0)
        except RuntimeError:
            members = [m.reshape(-1, m.shape[-1]) for m in members]
            preds = torch.stack(members, dim=0)

        # ensure target shape matches members' spatial/state dims
        N, d_state = preds.shape[1], preds.shape[2]
        if target.shape[0] != N or target.shape[1] != d_state:
            if target.numel() == N * d_state:
                target = target.reshape(N, d_state)
            else:
                raise ValueError(f"shape mismatch for {expname} {date}: preds {(preds.shape)} target {(target.shape)}")

        # ensemble mean (stays on device)
        ens_mean = torch.mean(preds, dim=0)  # (N, d_state)

        # weights for metrics on same device
        weights = torch.ones_like(ens_mean, device=device)

        # RMSE of ensemble mean using metrics.mse
        mse_val = metrics.mse(ens_mean, target, weights, mask=None, average_grid=True, sum_vars=True)
        rmse = math.sqrt(scalar_from_tensor(mse_val))

        # spread (std) from spread_squared (ens_dim=0)
        spread_var = metrics.spread_squared(preds, target, None, mask=None, average_grid=True, sum_vars=True, ens_dim=0)
        spread = math.sqrt(scalar_from_tensor(spread_var))

        spread_skill = spread / (rmse + 1e-12)

        # CRPS (ensemble estimator). use ens_dim=0
        crps_val = metrics.crps_ens(preds, target, None, mask=None, average_grid=True, sum_vars=True, ens_dim=0)
        crps_val = scalar_from_tensor(crps_val)

        results.append({"date": date, "rmse": rmse, "spread": spread, "spread_skill": spread_skill, "crps_ens": crps_val})

    df = pd.DataFrame(results)
    summary = {
        "exp": expname,
        "n_dates": len(df),
        "rmse_mean": float(df["rmse"].mean()) if not df.empty else float("nan"),
        "spread_mean": float(df["spread"].mean()) if not df.empty else float("nan"),
        "spread_skill_mean": float(df["spread_skill"].mean()) if not df.empty else float("nan"),
        "crps_ens_mean": float(df["crps_ens"].mean()) if not df.empty else float("nan"),
    }
    return df, summary


if __name__ == "__main__":
    all_summaries = []
    for exp in EXPERIMENTS:
        df, summary = compute_for_experiment(exp)
        print(summary)
        out_csv = Path(".") / f"metrics_{exp}.csv"
        df.to_csv(out_csv, index=False)
        all_summaries.append(summary)
    pd.DataFrame(all_summaries).to_csv("metrics_summary.csv", index=False)

In [ ]:
from pathlib import Path
expdir = Path("/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/280126/EDM_50")
date = "2009-12-29"
print(sorted(p.name for p in expdir.glob("*.pt"))[:50])
print("matches target_* date:", sorted(p.name for p in expdir.glob(f"*{date}*.pt")))

In [ ]:
import numpy as np

def destandardize(
    sample,
    pr_stats_path='/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.pr_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_mm_day_noleap.npy', 
    tas_stats_path='/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.tas_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_noleap.npy',
    std_dataset=False
):
    pr_mean, pr_std = np.load(pr_stats_path)
    tas_mean, tas_std = np.load(tas_stats_path)

    if std_dataset:
        return sample * np.array([pr_std, tas_std])
    else:
        return sample * np.array([pr_std, tas_std]) + np.array([pr_mean, tas_mean])
    
pr_mean, pr_std = np.load('/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.pr_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_mm_day_noleap.npy')
tas_mean, tas_std = np.load('/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.tas_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_noleap.npy')

# RMSE * std and CRPS * std
print(f"pr mean: {pr_mean}, pr std: {pr_std}")
print(f"tas mean: {tas_mean}, tas std: {tas_std}")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# paths to the mean/std files you already referenced
PR_STATS = Path("/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.pr_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_mm_day_noleap.npy")
TAS_STATS = Path("/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.tas_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_noleap.npy")

METRICS_DIR = Path(".")  # directory containing metrics_{exp}.csv
EXPERIMENTS = ["SI_100", "CorrDiff_50", "EDM_50"]


def _load_std(stats_path: Path):
    mean, std = np.load(stats_path)
    # if std is an array (per-channel), reduce to scalar using mean
    if np.ndim(std) > 0:
        std_scalar = float(np.mean(std))
    else:
        std_scalar = float(std)
    return std_scalar


def scale_metrics(var: str, out_dir: Path = METRICS_DIR):
    var = var.lower()
    if var not in ("pr", "tas"):
        raise ValueError("var must be 'pr' or 'tas'")

    stats_path = PR_STATS if var == "pr" else TAS_STATS
    std = _load_std(stats_path)

    scaled_summaries = []
    for exp in EXPERIMENTS:
        fn = METRICS_DIR / f"metrics_{exp}.csv"
        if not fn.exists():
            print(f"missing {fn}, skipping")
            continue
        df = pd.read_csv(fn)

        # scale requested metrics (if present)
        for col in ("rmse", "crps_ens"):
            if col in df.columns:
                df[f"{col}_scaled_{var}"] = df[col] * std

        # optional: scale spread too if present
        if "spread" in df.columns:
            df[f"spread_scaled_{var}"] = df["spread"] * std

        out_fn = out_dir / f"metrics_{exp}_scaled_{var}.csv"
        df.to_csv(out_fn, index=False)
        print(f"wrote {out_fn}")

        # build per-experiment summary
        summary = {
            "exp": exp,
            "n_dates": int(df.shape[0]),
            "rmse_mean_scaled": float(df[f"rmse_scaled_{var}"].mean()) if f"rmse_scaled_{var}" in df.columns else float("nan"),
            "crps_ens_mean_scaled": float(df[f"crps_ens_scaled_{var}"].mean()) if f"crps_ens_scaled_{var}" in df.columns else float("nan"),
        }
        scaled_summaries.append(summary)

    if scaled_summaries:
        pd.DataFrame(scaled_summaries).to_csv(out_dir / f"metrics_summary_scaled_{var}.csv", index=False)
        print(f"wrote summary metrics_summary_scaled_{var}.csv")


if __name__ == "__main__":
    # example: scale for precipitation then temperature
    scale_metrics("pr")
    scale_metrics("tas")

In [ ]:
# ...existing code...
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(".")
METRICS_DIR = ROOT  # where metrics_{exp}.csv live

# experiments are named MODEL_NFE (e.g. "SI_100", "CorrDiff_50", "EDM_50")
ENS_COUNT = 5  # all experiments have 5 ensemble members

# optional baseline (UNET) metrics file (leave if not available)
UNET_METRICS = METRICS_DIR / "metrics_UNET.csv"

# paths to stats
PR_STATS = Path("/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.pr_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_mm_day_noleap.npy")
TAS_STATS = Path("/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.tas_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_noleap.npy")

def load_std(path: Path):
    mean, std = np.load(path)
    # if std is array, reduce to scalar
    std_val = float(np.mean(std)) if np.ndim(std) > 0 else float(std)
    return std_val

pr_std = load_std(PR_STATS)
tas_std = load_std(TAS_STATS)

def human_label_from_exp(exp_name: str, ens: int = ENS_COUNT) -> str:
    # exp_name expected "MODEL_NFE" -> "MODEL NFE NFE <ens> ens"
    parts = exp_name.split("_", 1)
    if len(parts) == 2:
        model, nfe = parts
        return f"{model} {nfe} NFE {ens} ens"
    return f"{exp_name} {ens} ens"

def summarize_for_var(var: str, std: float, include_unet: bool = True):
    rows = []

    # optional UNET baseline first (print but keep SSR/CRPS as --)
    if include_unet and UNET_METRICS.exists():
        df_unet = pd.read_csv(UNET_METRICS)
        rmse_mean = df_unet["rmse"].mean() if "rmse" in df_unet.columns else float("nan")
        rmse_scaled = rmse_mean * std
        # baseline: no ensemble spread_skill or crps_ens -> show as NaN for formatting
        rows.append(("UNET", rmse_scaled, float("nan"), float("nan")))
    elif include_unet:
        # UNET file missing, still allow hardcoded placeholder if you want; skip here
        pass

    for exp in EXPERIMENTS:
        fn = METRICS_DIR / f"metrics_{exp}.csv"
        if not fn.exists():
            print(f"missing {fn}, skipping")
            continue
        df = pd.read_csv(fn)
        # per-date columns expected: rmse, spread, spread_skill, crps_ens
        rmse_mean = df["rmse"].mean() if "rmse" in df.columns else float("nan")
        crps_mean = df["crps_ens"].mean() if "crps_ens" in df.columns else float("nan")
        # prefer spread_skill column if present, else compute spread_mean / rmse_mean
        if "spread_skill" in df.columns:
            ssr = df["spread_skill"].mean()
        elif ("spread" in df.columns) and (rmse_mean != 0):
            ssr = df["spread"].mean() / (rmse_mean + 1e-12)
        else:
            ssr = float("nan")

        rmse_scaled = rmse_mean * std
        crps_scaled = crps_mean * std

        label = human_label_from_exp(exp)
        rows.append((label, rmse_scaled, ssr, crps_scaled))
    return rows

def fmt_row(label, rmse, ssr, crps):
    def fmt(x):
        if pd.isna(x):
            return "{--}"
        return f"{x:.3f}"
    return f"{label} & {fmt(rmse)} & {fmt(ssr)} & {fmt(crps)} \\\\"

# build LaTeX table parts
pre_rows = summarize_for_var("pr", pr_std, include_unet=True)
tas_rows = summarize_for_var("tas", tas_std, include_unet=True)

lines = []
lines.append(r"Model & {RMSE} & {SSR} & {CRPS} \\")
lines.append(r"\midrule")
lines.append("")
lines.append(r"\multicolumn{4}{l}{\textbf{Precipitation}} \\")
lines.append(r"\addlinespace[0.3em]")
for r in pre_rows:
    lines.append(fmt_row(*r))

lines.append("")
lines.append(r"\addlinespace[0.6em]")
lines.append(r"\multicolumn{4}{l}{\textbf{Temperature (2\,m)}} \\")
lines.append(r"\addlinespace[0.3em]")
for r in tas_rows:
    lines.append(fmt_row(*r))

lines.append(r"\bottomrule")

latex_table = "\n".join(lines)
print(latex_table)

# optionally save to file
(Path(".") / "metrics_table.tex").write_text(latex_table)
print("written metrics_table.tex")
# ...existing code...